In [1]:
pip install python-calamine


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import os

# Ruta del archivo original (usando r'' para evitar problemas con las barras invertidas de Windows)
file_path = r"C:\Users\gabri\Downloads\orderDetails - 2026-01-22T090739.886 (1).xlsx"

# 1. Cargar el archivo y gestionar los encabezados
# En Power Query hiciste dos "Promote Headers". Generalmente en Pandas esto significa que
# la data real empieza en la fila 2 (índice 1) o 3 (índice 2).
# Ajusta 'header=1' o 'header=2' si ves que los títulos no quedan bien a la primera.
# Según tu código M, parece que hay metadata al principio, así que asumimos que la data tabla real empieza un poco más abajo.
try:
    # Intento leer asumiendo que la primera fila es basura y la segunda son los headers reales (común en estos reportes)
    df = pd.read_excel(file_path, header=1, engine='calamine')
except Exception as e:
    print(f"Error leyendo el archivo: {e}")
    exit()

# 2. Limpieza de Tipos de Datos (Equivalente a Table.TransformColumnTypes)
# Convertimos columnas clave a numérico o fecha para evitar errores
# (Pandas suele detectar muchos automáticamente, pero forzamos los importantes)

# Columnas numéricas (rellenamos NaN con 0 si es necesario para cálculos)
cols_numericas = ['Total del pedido', 'Monto de pago', 'Comisión', 'Cargos']
for col in cols_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Columnas de fecha
cols_fechas = ['Fecha del pedido', 'Aceptado en', 'Entregado el', 'Cancelado el']
for col in cols_fechas:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 3. Filtrado Principal (Equivalente a #"Filas filtradas")
# Filtramos solo "Pago online"
df_limpio = df[df['Forma de pago'] == "Pago online"].copy()

# 4. Generar el reporte de Cancelados (Tu requerimiento adicional)
# Aquí tomamos la data ya filtrada (Online) y filtramos por "Cancelado".
# OJO: Si querías los cancelados de TODOS los pagos (incluyendo efectivo), cambia 'df_limpio' por 'df' abajo.
df_cancelados = df[df['Estado del pedido'].str.upper() == 'CANCELADO'].copy()

# 5. Exportar a Excel con múltiples hojas
output_path = r"C:\Users\gabri\Downloads\Reporte_Procesado.xlsx"

try:
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # Hoja 1: Data Limpia (Pago Online)
        df_limpio.to_excel(writer, sheet_name='Data Limpia', index=False)
        
        # Hoja 2: Solo Cancelados
        df_cancelados.to_excel(writer, sheet_name='Reporte Cancelados', index=False)
        
    print(f"Proceso finalizado con éxito. Archivo guardado en: {output_path}")

except Exception as e:
    print(f"No se pudo guardar el archivo: {e}")

Proceso finalizado con éxito. Archivo guardado en: C:\Users\gabri\Downloads\Reporte_Procesado.xlsx
